In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import os
import socket
import subprocess
import time

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

def wait_for_server(port, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        try:
            requests.get(f"http://localhost:{port}", timeout=1)
            return True
        except requests.ConnectionError:
            time.sleep(0.5)
    raise TimeoutError(f"Server didn't start on port {port} within {timeout}s")

PORT = find_free_port()
pwd = os.path.dirname(os.getcwd())

server = subprocess.Popen(
    ["npm", "run", "dev", "--", "--port", str(PORT)],
    cwd=pwd,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

try:
    wait_for_server(PORT)
    print(f"Server running on port {PORT}")

    searchable_pages = []

    for dir in ['cons', 'design', 'soro', 'signals', 'transfer']:
        dir_path = os.path.join(pwd, "src/pages", dir)
        for page in os.listdir(dir_path):
            if page.endswith(".astro"):  # skip images etc.
                searchable_pages.append(os.path.join(dir, page.replace(".astro", '')))

    page_text = []

    for page in searchable_pages:
        print(page)
        url = f"http://localhost:{PORT}/{page}"
        html = requests.get(url).text

        soup = BeautifulSoup(html, features="html.parser")

        for script in soup(["script", "style", "nav", 'header', 'footer']):
            script.extract()

        for el_id in ["nav_container", "page_title", "title_bar"]:
            el = soup.find(attrs={"id": el_id})
            if el:
                el.extract()

        text = soup.get_text()
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = ' '.join(chunk for chunk in chunks if chunk)

        banned_strings = [
            "Derivation +",
            "Solution +",
            "Scroll back to top"
        ]
        for b in banned_strings:
            text = text.replace(b, "")

        page_text.append({
            "title": page.split("/")[1].replace("_", " ").capitalize(),
            "text": text,
            'link': "/" + page,
            'course': page.split('/')[0]
        })

    json.dump(page_text, open("../src/search.json", "w"), indent=2)

finally:
    server.terminate()   
    print("Server stopped")

Server running on port 57222
cons/conservation_of_energy
cons/p1
cons/p3
cons/conservation_of_momentum
cons/problem_solving_approaches
cons/conservation_of_mass
cons/p2
cons/conservation_and_accounting_laws
cons/introduction_to_variables_and_properties
cons/worskshop_sample_Jenny
design/interviews
design/biodesign
design/team
design/soldering
design/icorps
design/documentation
design/cad
design/printing
design/underconstruction_page
design/hcd
design/laser
design/client
design/research
soro/prototyping
soro/mech_prop
soro/design_process
soro/actuators
soro/applications
soro/history
soro/mech_prog
soro/sensors
signals/time_domain
signals/introduction_to_signals_and_systems
signals/stability
signals/laplace
signals/underconstruction_page
signals/filters
signals/transfer_functions
signals/eigenvalues
signals/convolution
signals/fourier_transform
transfer/Mass_Equilibrium_and_Kinetics
transfer/Transport_in_a_biomedical_context
transfer/mass_ge_bc
transfer/Radiation
transfer/Diffusion
trans